In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
import os
import shutil
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import random
import cv2

In [ ]:
DATASET_PATH = "/content/drive/MyDrive/violence-detection/Dataset"
SPLITS_DIR = "/content/drive/MyDrive/violence-detection/splits"
CHECKPOINT_DIR = "/content/drive/MyDrive/violence-detection/checkpoints/"
RESULTS_DIR = "/content/drive/MyDrive/violence-detection/results"

In [ ]:
FRAMES = 30
IMG_SIZE = 224
BATCH_SIZE = 8
NUM_CLASSES = 2
EPOCHS = 40
LEARNING_RATE = 1e-4
CLASSES = ["Non-Violent", "Violent"]

In [ ]:
def create_or_get_splits(dataset_path, splits_dir, split_ratio=(0.7, 0.2, 0.1)):
    """
    Creates train/val/test folders from the dataset if not exist.
    Returns paths: X_train, X_val, X_test, y_train, y_val, y_test
    """
    train_dir = os.path.join(splits_dir, "train")
    val_dir = os.path.join(splits_dir, "val")
    test_dir = os.path.join(splits_dir, "test")

    if all(os.path.exists(p) for p in [train_dir, val_dir, test_dir]):
        print("Split folders exist. Using existing folders.")
    else:
        print("Creating split folders...")
        for split in ["train", "val", "test"]:
            for cls in ["violent", "non-violent"]:
                os.makedirs(os.path.join(splits_dir, split, cls), exist_ok=True)

        # Collect all videos per class
        class_videos = {}
        for cls in ["violent", "non-violent"]:
            cls_path = os.path.join(dataset_path, cls)
            class_videos[cls] = [os.path.join(cls_path, f) for f in os.listdir(cls_path)
                                 if f.lower().endswith(('.mp4', '.avi', '.mov'))]

        # Stratified split
        for cls, videos in class_videos.items():
            random.shuffle(videos)
            n = len(videos)
            n_train = int(split_ratio[0]*n)
            n_val = int(split_ratio[1]*n)
            train_videos = videos[:n_train]
            val_videos = videos[n_train:n_train+n_val]
            test_videos = videos[n_train+n_val:]

            for video in train_videos:
                shutil.copy(video, os.path.join(train_dir, cls))
            for video in val_videos:
                shutil.copy(video, os.path.join(val_dir, cls))
            for video in test_videos:
                shutil.copy(video, os.path.join(test_dir, cls))
        print("Split folders created.")

    # Return lists of video paths and labels
    def load_paths_labels(split_dir):
        X, y = [], []
        for i, cls in enumerate(["non-violent", "violent"]):
            cls_path = os.path.join(split_dir, cls)
            files = [os.path.join(cls_path, f) for f in os.listdir(cls_path)
                     if f.lower().endswith(('.mp4', '.avi', '.mov'))]
            X.extend(files)
            y.extend([i]*len(files))
        return X, y

    X_train, y_train = load_paths_labels(train_dir)
    X_val, y_val     = load_paths_labels(val_dir)
    X_test, y_test   = load_paths_labels(test_dir)

    return X_train, y_train, X_val, y_val, X_test, y_test

X_train, y_train, X_val, y_val, X_test, y_test = create_or_get_splits(DATASET_PATH, SPLITS_DIR)
print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

In [ ]:
def load_clip(video_path):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    step = max(total_frames // FRAMES, 1)
    clip = []
    for i in range(FRAMES):
        frame_idx = i * step
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        clip.append(frame.astype(np.float32)/255.0)
    cap.release()
    while len(clip) < FRAMES:
        clip.append(clip[-1])
    return np.array(clip, dtype=np.float32)

In [ ]:
def augment_clip(clip):
    if tf.random.uniform(()) > 0.5:
        clip = tf.image.flip_left_right(clip)
    clip = tf.image.random_brightness(clip, max_delta=0.2)
    clip = tf.clip_by_value(clip, 0.0, 1.0)
    return clip

def preprocess_path(video_path, label, augment=False):
    clip = tf.numpy_function(load_clip, [video_path], tf.float32)
    clip.set_shape((FRAMES, IMG_SIZE, IMG_SIZE, 3))
    if augment:
        clip = augment_clip(clip)
    label_onehot = tf.one_hot(label, NUM_CLASSES)
    return clip, label_onehot

In [ ]:
def create_dataset(X, y, augment=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    ds = ds.shuffle(len(X)).map(lambda x,y: preprocess_path(x, y, augment=augment),
                                num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = create_dataset(X_train, y_train, augment=True)
val_ds   = create_dataset(X_val, y_val, augment=False)
test_ds  = create_dataset(X_test, y_test, augment=False)

In [ ]:
def build_model():
    base_cnn = tf.keras.applications.MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3),
                                                  include_top=False, weights='imagenet',
                                                  pooling='avg')
    base_cnn.trainable = False
    inputs = keras.Input(shape=(FRAMES, IMG_SIZE, IMG_SIZE, 3))
    x = layers.TimeDistributed(base_cnn)(inputs)
    x = layers.LSTM(128)(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer=keras.optimizers.Adam(LEARNING_RATE),
                  loss='categorical_crossentropy', metrics=['accuracy'])
    return model

model = build_model()
model.summary()

In [ ]:
# Save **all epoch checkpoints**
checkpoint_all = keras.callbacks.ModelCheckpoint(
    filepath=os.path.join(CHECKPOINT_DIR, "epoch_{epoch:02d}.weights.h5"),
    save_weights_only=True,
    save_best_only=False,   # save every epoch
    verbose=1
)

# Save **best model according to validation loss**
checkpoint_best = keras.callbacks.ModelCheckpoint(
    filepath=os.path.join(CHECKPOINT_DIR, "best_model.weights.h5"),
    monitor="val_loss",
    save_best_only=True,
    save_weights_only=True,
    verbose=1
)

# **Better early stopping**: consider small improvements and restore best weights
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    min_delta=1e-4,   # small improvement threshold
    patience=7,       # wait more epochs before stopping
    restore_best_weights=True,
    verbose=1
)

# Reduce LR if stuck
reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-7,
    verbose=1
)

# Optional: Stop if accuracy stabilizes
class AccuracyStabilityStopping(keras.callbacks.Callback):
    def __init__(self, monitor="val_accuracy", patience=5, decimals=3):
        super().__init__()
        self.monitor = monitor
        self.patience = patience
        self.decimals = decimals
        self.history = []

    def on_epoch_end(self, epoch, logs=None):
        current = logs.get(self.monitor)
        if current is None:
            return
        current_rounded = np.round(current, self.decimals)
        self.history.append(current_rounded)
        if len(self.history) >= self.patience:
            last_values = self.history[-self.patience:]
            if len(set(last_values)) == 1:
                print(f"\nStopping early: {self.monitor} stabilized at {current_rounded}")
                self.model.stop_training = True

accuracy_stable_stop = AccuracyStabilityStopping(
    monitor="val_accuracy",
    patience=5
)

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[checkpoint_all, checkpoint_best, early_stop, reduce_lr, accuracy_stable_stop]
)

In [ ]:
def plot_training(history, save_dir=RESULTS_DIR):
    plt.figure()
    plt.plot(history.history['accuracy'])
    plt.plot(history.history['val_accuracy'])
    plt.title("Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend(["Train", "Val"])
    plt.savefig(os.path.join(save_dir, "accuracy.png"))
    plt.show()

    plt.figure()
    plt.plot(history.history['loss'])
    plt.plot(history.history['val_loss'])
    plt.title("Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend(["Train", "Val"])
    plt.savefig(os.path.join(save_dir, "loss.png"))
    plt.show()

plot_training(history)

In [ ]:
import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc, precision_recall_curve, precision_recall_fscore_support
import matplotlib.pyplot as plt
import seaborn as sns
import json

In [ ]:
TEST_DIR = "/content/drive/MyDrive/violence-detection/splits/test"
CHECKPOINT_PATH = "/content/drive/MyDrive/violence-detection/checkpoint_8/epoch_16.weights.h5"
RESULTS_DIR = "/content/drive/MyDrive/violence-detection/test_results1"
os.makedirs(RESULTS_DIR, exist_ok=True)

FRAMES = 30
IMG_SIZE = 224
BATCH_SIZE = 1
NUM_CLASSES = 2
CLASSES = ["Non-Violent", "Violent"]

In [ ]:
print("Test dir exists:", os.path.exists(TEST_DIR))
print("Contents:", os.listdir("/content/drive/MyDrive/violence-detection/splits"))

In [ ]:
def build_dataset(test_dir):
    X_test = []
    y_test = []

    class_map = {"non-violent":0, "violent":1}

    for class_name in sorted(class_map.keys()):
        class_path = os.path.join(test_dir, class_name)

        if not os.path.exists(class_path):
            print("Missing:", class_path)
            continue

        files = sorted(os.listdir(class_path))

        for file in files:
            if file.lower().endswith(('.mp4','.avi','.mov')):
                X_test.append(os.path.join(class_path,file))
                y_test.append(class_map[class_name])

    return X_test, y_test

In [ ]:
def load_clip(video_path):
    if isinstance(video_path, np.ndarray):
        video_path = video_path.item()
    if isinstance(video_path, bytes):
        video_path = video_path.decode('utf-8')

    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames < FRAMES:
        step = 1
        n_frames = total_frames
    else:
        step = total_frames // FRAMES
        n_frames = FRAMES

    clip = []
    for i in range(n_frames):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i * step)
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        clip.append(frame.astype(np.float32)/255.0)

    cap.release()
    if len(clip) == 0:
        print(f"WARNING: Could not read video {video_path}")
        return np.zeros((FRAMES, IMG_SIZE, IMG_SIZE, 3), dtype=np.float32)

    while len(clip) < FRAMES:
        clip.append(clip[-1])

    return np.array(clip, dtype=np.float32)

In [ ]:
def build_model():
    base_cnn = keras.applications.MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights='imagenet',
        pooling='avg'
    )
    base_cnn.trainable = False

    inputs = keras.Input(shape=(FRAMES, IMG_SIZE, IMG_SIZE, 3))
    x = layers.TimeDistributed(base_cnn)(inputs)
    x = layers.LSTM(128)(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
    model = keras.Model(inputs, outputs)

    model.compile(
        optimizer=keras.optimizers.Adam(),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model = build_model()
model.load_weights(CHECKPOINT_PATH)
print("Model weights loaded.")

In [ ]:
y_true = []
y_pred_proba = []

print("Generating predictions...")
for i, video_path in enumerate(X_test):
    clip = np.expand_dims(load_clip(video_path), axis=0)
    prediction = model.predict(clip, verbose=0)
    y_true.append(y_test[i])
    y_pred_proba.append(prediction[0])
    if (i+1) % 10 == 0:
        print(f"Processed {i+1}/{len(X_test)} videos")

y_true = np.array(y_true)
y_pred_proba = np.array(y_pred_proba)
y_pred = np.argmax(y_pred_proba, axis=1)
accuracy = np.mean(y_pred == y_true)
cm = confusion_matrix(y_true, y_pred)

print(f"Overall Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

In [ ]:
def plot_confusion_matrix_detailed(cm, classes, save_path):
    fig, (ax1, ax2) = plt.subplots(1,2, figsize=(16,7))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes, ax=ax1)
    ax1.set_title(f'Confusion Matrix (Accuracy: {np.trace(cm)/np.sum(cm):.2%})')

    cm_percent = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
    annotations = np.empty_like(cm).astype(str)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            annotations[i,j] = f'{cm[i,j]}\n({cm_percent[i,j]:.1f}%)'
    sns.heatmap(cm_percent, annot=annotations, fmt='', xticklabels=classes, yticklabels=classes, ax=ax2, vmin=0, vmax=100)
    ax2.set_title('Confusion Matrix (Percentages)')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.show()
    plt.close()

plot_confusion_matrix_detailed(cm, CLASSES, os.path.join(RESULTS_DIR, "confusion_matrix_detailed.png"))


In [ ]:
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=range(len(CLASSES)))
plt.figure(figsize=(10,6))
x = np.arange(len(CLASSES))
width = 0.25
plt.bar(x-width, precision*100, width, label='Precision')
plt.bar(x, recall*100, width, label='Recall')
plt.bar(x+width, f1*100, width, label='F1')
plt.xticks(x, CLASSES)
plt.legend()
plt.title("Per-Class Metrics (%)")
plt.savefig(os.path.join(RESULTS_DIR, "per_class_metrics.png"))
plt.show()
plt.close()

In [ ]:
plt.figure(figsize=(8,6))
for i in range(len(CLASSES)):
    y_true_bin = (y_true == i).astype(int)
    fpr, tpr, _ = roc_curve(y_true_bin, y_pred_proba[:,i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{CLASSES[i]} (AUC={roc_auc:.3f})')
plt.plot([0,1],[0,1],'k--')
plt.legend()
plt.title("ROC Curve")
plt.savefig(os.path.join(RESULTS_DIR, "roc_curve.png"))
plt.show()
plt.close()


In [ ]:
plt.figure(figsize=(8,6))
for i in range(len(CLASSES)):
    y_true_bin = (y_true == i).astype(int)
    p, r, _ = precision_recall_curve(y_true_bin, y_pred_proba[:,i])
    plt.plot(r, p, label=CLASSES[i])
plt.legend()
plt.title("Precision-Recall Curve")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.savefig(os.path.join(RESULTS_DIR, "precision_recall_curve.png"))
plt.show()
plt.close()

In [ ]:
plt.figure(figsize=(8,6))
for i in range(len(CLASSES)):
    indices = np.where(y_true == i)[0]
    confidences = y_pred_proba[indices,i]*100
    plt.hist(confidences, bins=20, alpha=0.5, label=CLASSES[i])
plt.legend()
plt.title("Prediction Confidence Distribution")
plt.xlabel("Confidence (%)")
plt.ylabel("Number of Videos")
plt.savefig(os.path.join(RESULTS_DIR, "prediction_confidence_distribution.png"))
plt.show()
plt.close()

In [ ]:
with open(os.path.join(RESULTS_DIR, "detailed_test_results.txt"), "w") as f:
    f.write(classification_report(y_true, y_pred, target_names=CLASSES))

# Save JSON results
results_dict = {
    "accuracy": float(accuracy),
    "confusion_matrix": cm.tolist()
}
with open(os.path.join(RESULTS_DIR, "test_results.json"), "w") as f:
    json.dump(results_dict, f, indent=4)

print(f"\nAll test results saved to {RESULTS_DIR}")